<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/Proyectos%20finales/Grupo%204/Grupo_4_Esmeraldas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Grupo 4. Sector esmeraldero colombiano: métricas de comercio exterior para una hoja de ruta

**Asignatura:** Inteligencia en Negocios Globales — Universidad EAN
**Proyecto final — Grupo 4**
**Producto:** Esmeraldas talladas, **HS 710391** (rubíes, zafiros y esmeraldas: el código no aísla la esmeralda, como advierte la ficha). Estructura comercial con el **capítulo 71**.
**Ficha técnica de referencia:** *Ficha técnica del estudio — Sector esmeraldero* (2026). La serie 2016-2025 por destino se descargó de Trade Map para este cuaderno. Los tres reportes EMIS no se redistribuyen: el cuaderno los pide al ejecutarse.

---

## La pregunta del equipo

> ¿Cómo reducir la vulnerabilidad por concentración de destinos, enfrentar la presión competitiva de otros productores y acceder a segmentos de mayor valor agregado?

## Lo que calcula este cuaderno

Cada sección corresponde a un análisis que el equipo propuso en su ficha técnica. El cuaderno **calcula y grafica; no interpreta**. La interpretación es el trabajo del equipo.

| Métrica o análisis | Pregunta que responde |
|---|---|
| **Participación por destino** | ¿A quién le vende Colombia sus esmeraldas talladas? |
| **HHI, Theil y número equivalente** | ¿Qué tan concentrados están los destinos y cómo ha cambiado desde 2016? |
| **Crecimiento exportador** | ¿Crecen o caen las exportaciones año a año? |
| **RCA, RSCA y NRCA** | ¿Colombia, Brasil y Etiopía están especializados en HS 710391? |
| **Apertura comercial e IBCR de los mercados** | ¿Qué tan abiertas son la UE, Emiratos, Singapur y Suiza? |
| **Grubel-Lloyd del capítulo 71** | ¿El comercio de piedras y metales preciosos es interindustrial o intraindustrial? |
| **Crecimiento de importaciones y participación de Colombia** | ¿En qué destinos crece la demanda y cuánto de ella es colombiana? |
| **Ficha financiera EMIS** | ¿Cómo les va a las comercializadoras esmeralderas? |

## Cómo usar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo (`Entorno de ejecución → Ejecutar todas`). Viene en modo `"github"`: descarga sus propios datos del repositorio del curso, no tienes que subir nada.
2. Después de cada gráfica hay una celda que dice **Análisis del equipo**. Haz doble clic sobre ella y escribe la interpretación. Puedes agregar más celdas de texto donde quieras.
3. Las celdas marcadas **editable** contienen pesos o puntajes que el equipo debe ajustar con su propio criterio. Cámbialos y vuelve a ejecutar.
4. Al terminar: `Archivo → Descargar → Descargar .ipynb` y sube el archivo al aula virtual.

Todas las tablas y figuras se guardan además en la carpeta `salidas/` (panel izquierdo de Colab) para que las uses en el informe.

---
# 0. Preparación del entorno

In [ ]:
import os
import io
import csv
import glob
import time
import shutil
import zipfile
import urllib.request
import urllib.error
import urllib.parse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Paleta de colores del curso (validada para lectura accesible en pantalla e impresion)
AZUL      = "#2a78d6"    # serie principal / pais del caso
ROJO      = "#e34948"    # valores negativos (par divergente con el azul)
NARANJA   = "#eb6834"    # elemento destacado
VERDE     = "#3f8f6b"    # segunda serie categorica
GRIS_MID  = "#c3c2b7"    # resto / punto neutro
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

# Orden fijo de colores para series categoricas: nunca se reciclan, lo que sobra va a "Otros" en gris
CATEGORIAS = [AZUL, NARANJA, VERDE, ROJO]

CARPETA_SALIDA = "salidas"
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def estilo(ax, titulo, subtitulo="", fuente="", eje_y="", eje_x="", rejilla="y"):
    """Aplica el estilo de graficas del curso: titulo a la izquierda, sin marco, rejilla suave, fuente al pie."""
    ax.set_title(titulo + ("\n" + subtitulo if subtitulo else ""),
                 fontsize=13, color=TINTA, loc="left", pad=14)
    ax.set_ylabel(eje_y, fontsize=10, color=GRIS_TEXT)
    ax.set_xlabel(eje_x, fontsize=10, color=GRIS_TEXT)
    if rejilla == "y":
        ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
    elif rejilla == "x":
        ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(GRIS_EJE)
    ax.tick_params(colors=GRIS_EJE, labelsize=9)
    if fuente:
        ax.figure.text(0.01, -0.03, "Fuente: " + fuente, fontsize=8, color=GRIS_EJE, ha="left")


def guardar(fig, nombre):
    """Muestra la figura y la guarda como PNG en la carpeta de salidas."""
    ruta = os.path.join(CARPETA_SALIDA, nombre + ".png")
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada en", ruta)


def exportar(tabla, nombre):
    """Guarda una tabla como CSV en la carpeta de salidas y la devuelve para mostrarla."""
    tabla.to_csv(os.path.join(CARPETA_SALIDA, nombre + ".csv"), index=False, encoding="utf-8-sig")
    return tabla


def fmt_miles(x, _=None):
    """Formato de eje para valores en miles de USD: 1200 -> 1.2 mn ; 850 -> 850 k."""
    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f} mil mn"
    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f} mn"
    return f"{x:.0f} k"


def fmt_unidades(x, _=None):
    """Formato de eje para valores en unidades: 1.5e9 -> 1.5 mil mn ; 2.4e6 -> 2.4 mn ; 850000 -> 850 k."""
    if abs(x) >= 1e9:
        return f"{x / 1e9:.1f} mil mn"
    if abs(x) >= 1e6:
        return f"{x / 1e6:.1f} mn"
    if abs(x) >= 1e3:
        return f"{x / 1e3:.0f} k"
    return f"{x:.0f}"


def leyenda_fuera(ax):
    """Leyenda a la derecha del grafico, para que no tape barras ni la nota de fuente."""
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1))


print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: cambia solo esta celda si lo necesitas
# ============================================================

MODO = "github"         # opciones: "github" | "subir" | "local"

REPO_CURSO = "YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales"
RAMA = "main"

# Archivos que necesita este cuaderno, con su ruta dentro del repositorio del curso.
# Los del grupo estan en "Proyectos finales/Grupo 4/datos/"; los demas son datos del curso en "data/".
ARCHIVOS_REPO = {
    "exp_2025": "Proyectos finales/Grupo 4/datos/EXPORTACIONES DE COLOMBIA AL MUNDO.csv",
    "brasil_2025": "Proyectos finales/Grupo 4/datos/EXPORTACIONES DE BRAZIL HACIA EL MUNDO.csv",
    "mundo_exp": "Proyectos finales/Grupo 4/datos/EXPORTACIONES DEL MUNDO.csv",
    "cap71_exp": "Proyectos finales/Grupo 4/datos/EXPORTACIONES COLOMBIA HACIA EL MUNDO CAPITULO 71.csv",
    "cap71_imp": "Proyectos finales/Grupo 4/datos/IMPORTACIONES COLOMBIA HACIA EL MUNDO CAPITULO 71.csv",
    "exp_serie": "Proyectos finales/Grupo 4/datos/colombias-exports-to-world-by-importer_710391.csv",
    "wb_pib": "data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_234.csv",
    "wb_exp": "data/API_NE.EXP.GNFS.CD_DS2_en_csv_v2_35140.csv",
    "wb_imp": "data/API_NE.IMP.GNFS.CD_DS2_en_csv_v2_33330.csv",
    "canasta_co_exp": "data/co_exp_productos_hs2_serie.xls",
    "canasta_co_imp": "data/co_imp_productos_hs2_serie.csv",
    "canasta_mundo_exp": "data/mundo_exp_productos_hs2_serie.csv",
}

RUTA_LOCAL = "../.."    # solo si MODO = "local": raiz del repositorio, corriendo desde la carpeta del grupo

print(f"Modo seleccionado: {MODO}  |  {len(ARCHIVOS_REPO)} archivos")

In [ ]:
# ============================================================
#  Ejecuta esta celda tal cual: consigue los datos segun el modo
# ============================================================

def pedir(url, intentos=5):
    """Descarga una URL, reintentando si el servidor pide esperar (HTTP 429 de GitHub en Colab)."""
    for intento in range(intentos):
        try:
            peticion = urllib.request.Request(url, headers={"User-Agent": "cuaderno-ean"})
            with urllib.request.urlopen(peticion, timeout=90) as respuesta:
                return respuesta.read()
        except urllib.error.HTTPError as error:
            if error.code not in (403, 429, 500, 502, 503) or intento == intentos - 1:
                raise
            espera = int(error.headers.get("Retry-After") or 0) or 2 ** intento
            print(f"    servidor ocupado (HTTP {error.code}); reintento en {espera} s")
            time.sleep(espera)
        except urllib.error.URLError:
            if intento == intentos - 1:
                raise
            time.sleep(2 ** intento)


def descargar_datos(rutas_repo, destino):
    """Trae los archivos del repositorio a la carpeta destino, en una sola peticion (zip del repo).

    Cada archivo se guarda por su nombre, sin carpetas. Si ya existe no se vuelve a bajar.
    Si el zip falla, baja los archivos uno por uno desde raw.githubusercontent.com.
    """
    os.makedirs(destino, exist_ok=True)
    faltan = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if not faltan:
        print(f"Los {len(rutas_repo)} archivos ya estaban descargados.")
        return
    try:
        print(f"Descargando {len(faltan)} archivos en una sola peticion...\n")
        comprimido = pedir(f"https://codeload.github.com/{REPO_CURSO}/zip/refs/heads/{RAMA}")
        with zipfile.ZipFile(io.BytesIO(comprimido)) as paquete:
            for miembro in paquete.namelist():
                relativo = miembro.split("/", 1)[1] if "/" in miembro else miembro
                if relativo in faltan:
                    with paquete.open(miembro) as origen, \
                         open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                        shutil.copyfileobj(origen, salida)
                    print(f"  extraido: {os.path.basename(relativo)}")
    except Exception as error:
        print(f"\n  El paquete fallo ({type(error).__name__}). Voy archivo por archivo.\n")
        for relativo in faltan:
            url = f"https://raw.githubusercontent.com/{REPO_CURSO}/{RAMA}/" + urllib.parse.quote(relativo)
            with open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                salida.write(pedir(url))
            print(f"  descargado: {os.path.basename(relativo)}")
            time.sleep(0.5)
    perdidos = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if perdidos:
        raise FileNotFoundError(f"No se pudieron descargar: {perdidos}. Espera un minuto y vuelve a ejecutar.")


if MODO == "github":
    RUTA_BASE = "datos_crudos"
    descargar_datos(list(ARCHIVOS_REPO.values()), RUTA_BASE)

    def ruta(clave):
        return os.path.join(RUTA_BASE, os.path.basename(ARCHIVOS_REPO[clave]))

elif MODO == "subir":
    from google.colab import files
    print("Sube estos archivos:\n  " + "\n  ".join(os.path.basename(v) for v in ARCHIVOS_REPO.values()))
    files.upload()
    RUTA_BASE = "/content"

    def ruta(clave):
        encontrados = glob.glob(os.path.join(RUTA_BASE, "**", os.path.basename(ARCHIVOS_REPO[clave])), recursive=True)
        if not encontrados:
            raise FileNotFoundError(f"Falta el archivo {os.path.basename(ARCHIVOS_REPO[clave])}")
        return encontrados[0]

else:  # local
    RUTA_BASE = RUTA_LOCAL

    def ruta(clave):
        return os.path.join(RUTA_BASE, ARCHIVOS_REPO[clave])

for clave in ARCHIVOS_REPO:
    estado = "ok" if os.path.exists(ruta(clave)) else "FALTA"
    print(f"  {estado:5s} {clave:14s} -> {os.path.basename(ARCHIVOS_REPO[clave])}")

In [ ]:
# ============================================================
#  Lectores: cada funcion resuelve las trampas de un tipo de archivo
# ============================================================

def a_numero(serie):
    """Convierte a numero una columna que viene como texto (miles con coma, simbolos, espacios)."""
    return pd.to_numeric(
        pd.Series(serie).astype(str)
                        .str.replace(",", "", regex=False)
                        .str.replace(r"[^0-9.\-]", "", regex=True)
                        .replace("", np.nan),
        errors="coerce",
    )


# Nombres cortos en espanol para las columnas de Trade Map
COLUMNAS_TM = {
    "Value (kUSD)": "valor_kusd",
    "Balance (kUSD)": "balanza_kusd",
    "Quantity": "cantidad",
    "Quantity Unit": "unidad_cantidad",
    "Unit Value": "valor_unitario",
    "Unit Value Unit": "unidad_valor_unitario",
    "Share (%)": "participacion_pct",
    "Share Partner Country (%)": "cuota_en_socio_pct",
    "Share World (%)": "participacion_mundial_pct",
    "Ranking Partners": "ranking_socio",
    "Growth Value 5Y (%)": "crec_valor_5a_pct",
    "Growth Value 2Y (%)": "crec_valor_2a_pct",
    "Growth Value Partners 5Y (%)": "crec_importaciones_socio_5a_pct",
    "Growth Quantity 5Y (%)": "crec_cantidad_5a_pct",
}

AGREGADOS_NO_PAIS = ["Zona franca", "Zonas francas", "Áreas Nes", "Areas, nes", "Zona Nep", "Free Zones"]


def leer_trademap(ruta_archivo):
    """Lee una tabla de indicadores de Trade Map (corte de un anio, formato largo).

    Trampas que resuelve: una columna sin nombre en el encabezado, codigos de pais con cero
    inicial que pandas convertiria a entero, valor unitario con 28 decimales, y la fila
    "Mundo" mezclada con los paises. La columna `pais` queda lista para usar.
    """
    tabla = pd.read_csv(ruta_archivo, dtype={"reporterCd": str, "partnerCd": str, "productCd": str},
                        encoding="utf-8-sig")
    return limpiar_trademap(tabla)


def limpiar_trademap(tabla):
    """Limpieza comun a toda tabla de indicadores de Trade Map, venga de CSV o de Excel."""
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    for columna in ("reporterCd", "partnerCd"):
        tabla[columna] = tabla[columna].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(3)
    tabla = tabla.rename(columns=COLUMNAS_TM)
    for columna in COLUMNAS_TM.values():
        if columna in tabla.columns and columna not in ("unidad_cantidad", "unidad_valor_unitario"):
            tabla[columna] = a_numero(tabla[columna])
    if "valor_unitario" in tabla.columns:
        tabla["valor_unitario"] = tabla["valor_unitario"].round(2)
    # Si todos los socios son "Mundo", la tabla es una lista de paises (reporter); si no, es una lista de socios
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    return tabla


def separar_mundo(tabla):
    """Devuelve (fila Mundo, tabla solo con paises). Excluye agregados que no son paises."""
    es_mundo = tabla["codigo"] == "000"
    mundo = tabla[es_mundo].iloc[0]
    paises = tabla[~es_mundo & ~tabla["pais"].isin(AGREGADOS_NO_PAIS)].reset_index(drop=True)
    return mundo, paises


def leer_serie_trademap(ruta_archivo):
    """Lee la serie anual por socio (un anio por columna) y la devuelve en formato largo.

    Columnas de salida: codigo, pais, anio, valor_kusd. Incluye la fila Mundo (codigo 000).
    """
    tabla = pd.read_csv(ruta_archivo, dtype=str, encoding="utf-8-sig")
    columnas_anio = [c for c in tabla.columns if c[:4].isdigit()]
    tabla = tabla.rename(columns={c: c[:4] for c in columnas_anio})
    anios = [c[:4] for c in columnas_anio]
    for anio in anios:
        tabla[anio] = a_numero(tabla[anio])
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    largo = tabla.melt(id_vars=["codigo", "pais"], value_vars=anios, var_name="anio", value_name="valor_kusd")
    largo["anio"] = largo["anio"].astype(int)
    return largo.sort_values(["anio", "valor_kusd"], ascending=[True, False]).reset_index(drop=True)


def leer_banco_mundial(ruta_archivo, nombre_indicador):
    """Lee un archivo del Banco Mundial: cuatro filas de metadatos antes del encabezado y un anio por columna."""
    tabla = pd.read_csv(ruta_archivo, skiprows=4, encoding="utf-8-sig")
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    anios = [c for c in tabla.columns if c.isdigit()]
    largo = tabla.melt(id_vars=["Country Name", "Country Code"], value_vars=anios,
                       var_name="anio", value_name=nombre_indicador)
    largo["anio"] = largo["anio"].astype(int)
    return largo.rename(columns={"Country Name": "pais", "Country Code": "iso3"})


ANIOS_CANASTA = [str(a) for a in range(2021, 2026)]


def leer_canasta(ruta_archivo):
    """Lee la canasta por capitulo HS del curso (97 capitulos + TOTAL, 2021-2025).

    Funciona igual si el archivo es un .csv o un .xls que en realidad es HTML.
    Trampas: apostrofe delante del codigo y miles separados por coma como texto.
    """
    with open(ruta_archivo, encoding="utf-8", errors="ignore") as f:
        primeras_letras = f.read(200).lstrip().lower()
    if primeras_letras.startswith("<"):
        tabla = max(pd.read_html(ruta_archivo), key=lambda t: t.shape[0])
    else:
        tabla = pd.read_csv(ruta_archivo)
    tabla = tabla.iloc[:, -7:]
    tabla.columns = ["codigo", "producto"] + ANIOS_CANASTA
    tabla["codigo"] = tabla["codigo"].astype(str).str.strip().str.lstrip("'").str.strip()
    for anio in ANIOS_CANASTA:
        tabla[anio] = a_numero(tabla[anio])
    return tabla.dropna(subset=["2025"]).reset_index(drop=True)


def fila_canasta(canasta, codigo):
    """Devuelve la serie 2021-2025 (en USD miles) de un codigo de la canasta: un capitulo HS2 o "TOTAL"."""
    fila = canasta[canasta["codigo"] == codigo]
    if fila.empty:
        raise KeyError(f"No encontre el codigo {codigo} en la canasta")
    return fila.iloc[0][ANIOS_CANASTA].astype(float)


print("Lectores listos.")

In [ ]:
# ============================================================
#  Formulas: las mismas de los cuadernos 2, 3 y 4 del curso
# ============================================================

def hhi(valores):
    """Indice de Herfindahl-Hirschman: suma de participaciones al cuadrado. Entre 0 y 1."""
    valores = np.asarray(valores, dtype=float)
    valores = valores[~np.isnan(valores)]
    valores = valores[valores > 0]
    if valores.sum() == 0:
        return np.nan
    participaciones = valores / valores.sum()
    return float(np.sum(participaciones ** 2))


def numeros_equivalentes(indice_hhi):
    """Cuantos destinos del mismo tamano equivaldrian a esta reparticion: 1 / HHI."""
    return 1 / indice_hhi

def cr_n(valores, n):
    """Razon de concentracion CR_n: participacion conjunta (%) de los n mayores."""
    valores = np.sort(np.asarray(valores, dtype=float))[::-1]
    valores = valores[~np.isnan(valores)]
    return float(valores[:n].sum() / valores.sum() * 100)

def theil(valores):
    """Indice de Theil: dispersion por entropia. 0 = reparto perfectamente igual; sin techo superior."""
    valores = np.asarray(valores, dtype=float)
    valores = valores[valores > 0]
    n = len(valores)
    media = valores.mean()
    return float((1 / n) * np.sum((valores / media) * np.log(valores / media)))

def grubel_lloyd(exportaciones, importaciones):
    """Indice de Grubel-Lloyd: 1 = comercio intraindustrial puro, 0 = interindustrial puro."""
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones
    return np.where(comercio_total > 0, 1 - np.abs(exportaciones - importaciones) / comercio_total, np.nan)

def ibcr(exportaciones, importaciones):
    """Indice de Balanza Comercial Relativa: (X - M) / (X + M), entre -1 y +1."""
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones
    return np.where(comercio_total > 0, (exportaciones - importaciones) / comercio_total, np.nan)

def apertura_comercial(exportaciones_totales, importaciones_totales, pib):
    """Indice de Apertura Comercial: (X + M) / PIB x 100, con los flujos TOTALES de la economia."""
    pib = np.asarray(pib, dtype=float)
    return np.where(pib > 0, (np.asarray(exportaciones_totales, dtype=float) + np.asarray(importaciones_totales, dtype=float)) / pib * 100, np.nan)

def rca_balassa(x_pais, x_pais_total, x_mundo, x_mundo_total):
    """Ventaja Comparativa Revelada de Balassa (1965). Mayor que 1 = especializacion revelada."""
    participacion_pais  = np.asarray(x_pais, dtype=float)  / np.asarray(x_pais_total, dtype=float)
    participacion_mundo = np.asarray(x_mundo, dtype=float) / np.asarray(x_mundo_total, dtype=float)
    return participacion_pais / participacion_mundo


def rsca_laursen(rca):
    """RCA simetrico de Laursen: (RCA - 1) / (RCA + 1), entre -1 y +1, con 0 como punto neutro."""
    rca = np.asarray(rca, dtype=float)
    return (rca - 1) / (rca + 1)

def nrca(x_pais, x_pais_total, x_mundo, x_mundo_total):
    """NRCA de Yu, Cai y Leung (2009): distancia entre la exportacion observada y la neutral. Suma cero."""
    x_pais, x_mundo = np.asarray(x_pais, dtype=float), np.asarray(x_mundo, dtype=float)
    x_pais_total, x_mundo_total = float(x_pais_total), float(x_mundo_total)
    observado = x_pais / x_mundo_total
    neutral   = (x_pais_total * x_mundo) / (x_mundo_total ** 2)
    return observado - neutral

def cagr(valor_inicial, valor_final, anios):
    """Tasa de crecimiento anual compuesto (%) entre dos valores separados por `anios` anios."""
    valor_inicial, valor_final = float(valor_inicial), float(valor_final)
    if valor_inicial <= 0 or valor_final <= 0 or anios <= 0:
        return np.nan
    return ((valor_final / valor_inicial) ** (1 / anios) - 1) * 100

print('Formulas listas.')

---
# 1. Los datos

Cinco CSV de Trade Map del equipo (cortes de 2025), la serie 2016-2025 por destino, tres archivos del Banco Mundial (PIB, exportaciones e importaciones totales) y la canasta HS2 del curso.

Dos trampas de los archivos del equipo: en `EXPORTACIONES DE COLOMBIA AL MUNDO.csv` la cantidad viene en cero para todos los destinos (el valor unitario no es calculable), y en el archivo de importaciones del capítulo 71 la columna `Balance` repite el balance de las exportaciones, así que solo se usa `Value`.

In [ ]:
exp_2025   = leer_trademap(ruta("exp_2025"))       # Colombia exporta 710391 por destino, 2025
brasil     = leer_trademap(ruta("brasil_2025"))    # Brasil exporta 710391 por destino, 2025
mundo_exp  = leer_trademap(ruta("mundo_exp"))      # exportadores mundiales de 710391, 2025
cap71_exp  = leer_trademap(ruta("cap71_exp"))      # Colombia exporta capitulo 71 por destino, 2025
cap71_imp  = leer_trademap(ruta("cap71_imp"))      # Colombia importa capitulo 71 por origen, 2025
serie      = leer_serie_trademap(ruta("exp_serie"))
wb_pib = leer_banco_mundial(ruta("wb_pib"), "pib_usd")
wb_exp = leer_banco_mundial(ruta("wb_exp"), "exportaciones_usd")
wb_imp = leer_banco_mundial(ruta("wb_imp"), "importaciones_usd")
canasta_co_exp, canasta_co_imp, canasta_mundo_exp = (leer_canasta(ruta(k)) for k in ("canasta_co_exp", "canasta_co_imp", "canasta_mundo_exp"))

mundo_2025, destinos_2025 = separar_mundo(exp_2025)
mundo_oferta, exportadores = separar_mundo(mundo_exp)
print(f"Colombia exporto HS 710391 por USD {mundo_2025['valor_kusd']:,.0f} miles en 2025 a {len(destinos_2025)} destinos (se excluye la fila 'Zona franca')")
print(f"El mundo exporto HS 710391 por USD {mundo_oferta['valor_kusd']:,.0f} miles; Colombia es el exportador n.º {int(exportadores.reset_index().index[exportadores['pais'] == 'Colombia'][0]) + 1}")
destinos_2025[["pais", "valor_kusd", "participacion_pct", "cuota_en_socio_pct", "ranking_socio", "crec_valor_5a_pct", "crec_importaciones_socio_5a_pct"]].head(10)

---
# 2. Participación por destino

In [ ]:
participacion = destinos_2025.nlargest(12, "valor_kusd")[["pais", "valor_kusd", "participacion_pct"]]
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = participacion.sort_values("participacion_pct")
ax.barh(orden["pais"], orden["participacion_pct"], color=AZUL, height=0.6)
for y, v in enumerate(orden["participacion_pct"]):
    ax.annotate(f"{v:.1f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Participación de cada destino en las exportaciones colombianas de HS 710391, 2025", "Doce mayores destinos",
       "Trade Map (ITC), 2025.", eje_x="% del valor exportado", rejilla="x")
guardar(fig, "g4_participacion_2025")
exportar(participacion, "g4_participacion_2025")

In [ ]:
paises_serie = serie[serie["codigo"] != "000"]
paises_serie = paises_serie[~paises_serie["pais"].isin(AGREGADOS_NO_PAIS)]
top4 = paises_serie[paises_serie["anio"] == 2025].nlargest(4, "valor_kusd")["pais"].tolist()
series_top = (paises_serie.assign(grupo=lambda d: np.where(d["pais"].isin(top4), d["pais"], "Otros"))
                          .groupby(["anio", "grupo"])["valor_kusd"].sum().unstack("grupo")[top4 + ["Otros"]])
exportar(series_top.reset_index(), "g4_series_destinos")
fig, ax = plt.subplots(figsize=(10, 5))
for color, columna in zip(CATEGORIAS + [GRIS_MID], series_top.columns):
    ax.plot(series_top.index, series_top[columna], color=color, linewidth=2.2, marker="o", markersize=5, label=columna)
ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
ax.set_xticks(series_top.index)
ax.legend(frameon=False, fontsize=9, loc="upper right")
estilo(ax, "Exportaciones colombianas de HS 710391 por destino, 2016-2025", "USD; cuatro mayores destinos de 2025 y el resto", "Trade Map (ITC), 2025.", eje_y="USD")
guardar(fig, "g4_series_destinos")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 3. Concentración: HHI, Theil y número equivalente, 2016-2025

In [ ]:
concentracion = (paises_serie.groupby("anio")["valor_kusd"]
                 .agg(HHI=hhi, Theil=theil, CR3=lambda v: cr_n(v, 3), destinos_activos=lambda v: int((v > 0).sum()))
                 .reset_index())
concentracion["numero_equivalente"] = numeros_equivalentes(concentracion["HHI"])
exportar(concentracion, "g4_concentracion")
concentracion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].plot(concentracion["anio"], concentracion["HHI"], color=AZUL, linewidth=2.4, marker="o", markersize=7)
for _, f in concentracion.iterrows():
    axes[0].annotate(f"{f['HHI']:.3f}", (f["anio"], f["HHI"]), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9, color=GRIS_TEXT)
axes[0].axhspan(0.18, 1, color=ROJO, alpha=0.06)
axes[0].axhspan(0.15, 0.18, color=NARANJA, alpha=0.06)
axes[0].set_ylim(0, max(0.6, concentracion["HHI"].max() * 1.2))
axes[0].set_xticks(concentracion["anio"])
estilo(axes[0], "HHI de destinos", "Bandas: > 0,18 concentrado; 0,15-0,18 moderado; < 0,15 fragmentado", eje_y="HHI")
axes[1].plot(concentracion["anio"], concentracion["Theil"], color=VERDE, linewidth=2.4, marker="o", markersize=7)
for _, f in concentracion.iterrows():
    axes[1].annotate(f"{f['Theil']:.2f}", (f["anio"], f["Theil"]), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9, color=GRIS_TEXT)
axes[1].set_ylim(0, concentracion["Theil"].max() * 1.25)
axes[1].set_xticks(concentracion["anio"])
estilo(axes[1], "Índice de Theil de destinos", "0 = reparto igual entre destinos activos; sube con la desigualdad", "Trade Map (ITC), 2025.", eje_y="Theil")
plt.tight_layout()
guardar(fig, "g4_concentracion")

**Análisis del equipo:** HHI y Theil no siempre cuentan la misma historia. ¿Coinciden aquí?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 4. Crecimiento exportador

In [ ]:
total = serie[serie["codigo"] == "000"].set_index("anio")["valor_kusd"]
crecimiento = pd.DataFrame({"valor_kusd": total})
crecimiento["variacion_interanual_pct"] = crecimiento["valor_kusd"].pct_change() * 100
print(f"CAGR 2016-2025: {cagr(total[2016], total[2025], 9):+.1f} %   |   2020-2025: {cagr(total[2020], total[2025], 5):+.1f} %")
print(f"Trade Map reporta para 2025: crecimiento 5 años {mundo_2025['crec_valor_5a_pct']:+.0f} %, 2 años {mundo_2025['crec_valor_2a_pct']:+.0f} %")
exportar(crecimiento.reset_index(), "g4_crecimiento")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].bar(total.index.astype(str), total.values, color=AZUL, width=0.6)
for x, v in enumerate(total.values):
    axes[0].annotate(fmt_miles(v), (x, v), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=8.5, color=GRIS_TEXT)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
estilo(axes[0], "Exportaciones colombianas de HS 710391", "USD, 2016-2025", eje_y="USD")
var = crecimiento["variacion_interanual_pct"].dropna()
axes[1].bar(var.index.astype(str), var.values, color=[ROJO if v < 0 else AZUL for v in var.values], width=0.6)
for x, v in enumerate(var.values):
    axes[1].annotate(f"{v:+.0f} %", (x, v), textcoords="offset points", xytext=(0, 4 if v >= 0 else -12), ha="center", fontsize=8.5, color=GRIS_TEXT)
axes[1].axhline(0, color=GRIS_EJE, linewidth=0.8)
estilo(axes[1], "Variación interanual", "(X_t - X_t-1) / X_t-1 x 100", "Trade Map (ITC), 2025.", eje_y="%")
plt.tight_layout()
guardar(fig, "g4_crecimiento")
crecimiento

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 5. Ventaja comparativa revelada: RCA, RSCA y NRCA

Para comparar países se necesitan sus **exportaciones totales**, que no están en las descargas del equipo. Se usan las exportaciones de bienes y servicios del Banco Mundial (una aproximación: incluyen servicios) para el último año con dato de todos los países. Zambia, nombrado en la ficha, **no aparece en la descarga mundial de HS 710391**, así que no se puede calcular.

Para Colombia se calcula además la serie 2021-2025 del capítulo 71 completo con la canasta del curso.

In [ ]:
COMPARADORES = {"Colombia": "COL", "Brasil": "BRA", "Etiopía": "ETH", "Zambia": "ZMB"}
ANIO_WB = 2024
totales_wb = wb_exp[wb_exp["anio"] == ANIO_WB].set_index("iso3")["exportaciones_usd"] / 1000     # a USD miles
X_w = totales_wb["WLD"]
X_j = mundo_oferta["valor_kusd"]
filas = []
for pais, iso in COMPARADORES.items():
    fila = exportadores[exportadores["pais"] == pais]
    if fila.empty or pd.isna(totales_wb.get(iso)):
        filas.append({"pais": pais, "nota": "sin dato en Trade Map o Banco Mundial"}); continue
    X_ij, X_i = fila.iloc[0]["valor_kusd"], totales_wb[iso]
    r = float(rca_balassa(X_ij, X_i, X_j, X_w))
    filas.append({"pais": pais, "X_ij_710391": X_ij, "X_i_total": X_i, "RCA": r, "RSCA": float(rsca_laursen(r)), "NRCA_x1e4": float(nrca(X_ij, X_i, X_j, X_w)) * 1e4})
vcr = pd.DataFrame(filas)
exportar(vcr, "g4_vcr_comparadores")
vcr

In [ ]:
vcr_cap71 = pd.DataFrame({"anio": [int(a) for a in ANIOS_CANASTA]})
vcr_cap71["RCA_cap71"] = rca_balassa(fila_canasta(canasta_co_exp, "71"), fila_canasta(canasta_co_exp, "TOTAL"),
                                     fila_canasta(canasta_mundo_exp, "71"), fila_canasta(canasta_mundo_exp, "TOTAL"))
vcr_cap71["RSCA_cap71"] = rsca_laursen(vcr_cap71["RCA_cap71"])
exportar(vcr_cap71, "g4_vcr_cap71")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
con_dato = vcr.dropna(subset=["RSCA"])
axes[0].bar(con_dato["pais"], con_dato["RSCA"], color=[NARANJA if p == "Colombia" else AZUL for p in con_dato["pais"]], width=0.55)
for x, (v, r) in enumerate(zip(con_dato["RSCA"], con_dato["RCA"])):
    axes[0].annotate(f"RSCA {v:+.2f}\nRCA {r:.1f}", (x, v), textcoords="offset points", xytext=(0, 5 if v >= 0 else -24), ha="center", fontsize=9, color=GRIS_TEXT)
axes[0].axhline(0, color=GRIS_EJE, linewidth=0.8)
axes[0].set_ylim(-1.1, 1.3)
estilo(axes[0], f"RSCA en HS 710391, {ANIO_WB}", "Denominador: exportaciones de bienes y servicios (Banco Mundial)", eje_y="RSCA")
axes[1].plot(vcr_cap71["anio"], vcr_cap71["RCA_cap71"], color=VERDE, linewidth=2.2, marker="o", markersize=6)
for _, f in vcr_cap71.iterrows():
    axes[1].annotate(f"{f['RCA_cap71']:.2f}", (f["anio"], f["RCA_cap71"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=GRIS_TEXT)
axes[1].axhline(1, color=GRIS_EJE, linewidth=0.8, linestyle="--")
axes[1].set_xticks(vcr_cap71["anio"])
axes[1].set_ylim(0, vcr_cap71["RCA_cap71"].max() * 1.3)
estilo(axes[1], "RCA de Balassa del capítulo 71, Colombia", "Línea punteada = 1", "Trade Map (ITC), Banco Mundial (WDI), canasta HS2 del curso.", eje_y="RCA")
plt.tight_layout()
guardar(fig, "g4_vcr")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 6. Apertura comercial e IBCR de los mercados candidatos, 2020-2024

Con los tres archivos del Banco Mundial del curso. La Unión Europea se toma como agregado (`EUU`). Emiratos Árabes Unidos no tiene dato de 2024 en esta versión de los WDI.

In [ ]:
MERCADOS = {"EUU": "Unión Europea", "ARE": "Emiratos Árabes Unidos", "SGP": "Singapur", "CHE": "Suiza", "COL": "Colombia (referencia)"}
macro = (wb_pib.merge(wb_exp, on=["iso3", "pais", "anio"]).merge(wb_imp, on=["iso3", "pais", "anio"]))
macro = macro[macro["iso3"].isin(MERCADOS) & macro["anio"].between(2020, 2024)].copy()
macro["mercado"] = macro["iso3"].map(MERCADOS)
macro["apertura_pct"] = apertura_comercial(macro["exportaciones_usd"], macro["importaciones_usd"], macro["pib_usd"])
macro["IBCR"] = ibcr(macro["exportaciones_usd"], macro["importaciones_usd"])
tabla_macro = macro.pivot(index="anio", columns="mercado", values="apertura_pct")
exportar(macro[["mercado", "iso3", "anio", "pib_usd", "exportaciones_usd", "importaciones_usd", "apertura_pct", "IBCR"]], "g4_apertura_ibcr")
tabla_macro.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for color, mercado in zip(CATEGORIAS + [GRIS_MID], MERCADOS.values()):
    if mercado in tabla_macro.columns:
        axes[0].plot(tabla_macro.index, tabla_macro[mercado], color=color, linewidth=2.2, marker="o", markersize=5, label=mercado)
axes[0].set_xticks(tabla_macro.index)
axes[0].legend(frameon=False, fontsize=9)
estilo(axes[0], "Índice de apertura comercial", "(X + M) / PIB x 100", eje_y="%")
ultimo = macro.sort_values("anio").groupby("mercado").tail(1).set_index("mercado")
axes[1].bar(ultimo.index, ultimo["IBCR"], color=[ROJO if v < 0 else AZUL for v in ultimo["IBCR"]], width=0.55)
for x, (v, a) in enumerate(zip(ultimo["IBCR"], ultimo["anio"])):
    axes[1].annotate(f"{v:+.3f} ({a})", (x, v), textcoords="offset points", xytext=(0, 5 if v >= 0 else -12), ha="center", fontsize=9, color=GRIS_TEXT)
axes[1].axhline(0, color=GRIS_EJE, linewidth=0.8)
axes[1].set_ylim(-0.3, 0.3)
axes[1].tick_params(axis="x", labelsize=8)
estilo(axes[1], "IBCR de la economía completa, último año con dato", "(X - M) / (X + M)", "Banco Mundial (WDI), 2025.", eje_y="IBCR")
plt.tight_layout()
guardar(fig, "g4_apertura_ibcr")

**Análisis del equipo:** ¿Qué dice la apertura de Singapur o Emiratos sobre su papel como hubs de reexportación?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 7. Grubel-Lloyd del capítulo 71

In [ ]:
X71_2025 = separar_mundo(cap71_exp)[0]["valor_kusd"]
M71_2025 = separar_mundo(cap71_imp)[0]["valor_kusd"]
gl_2025 = float(grubel_lloyd(X71_2025, M71_2025))
print(f"Capitulo 71, 2025 (archivos del equipo): X = {X71_2025:,.0f}  M = {M71_2025:,.0f}  ->  GL = {gl_2025:.4f}")
gl_serie = pd.DataFrame({"anio": [int(a) for a in ANIOS_CANASTA],
                         "X_cap71": fila_canasta(canasta_co_exp, "71").values, "M_cap71": fila_canasta(canasta_co_imp, "71").values})
gl_serie["GL_cap71"] = grubel_lloyd(gl_serie["X_cap71"], gl_serie["M_cap71"])
gl_serie["IBCR_cap71"] = ibcr(gl_serie["X_cap71"], gl_serie["M_cap71"])
exportar(gl_serie, "g4_grubel_lloyd")
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(gl_serie["anio"], gl_serie["GL_cap71"], color=AZUL, linewidth=2.2, marker="o", markersize=6, label="Grubel-Lloyd")
ax.plot(gl_serie["anio"], gl_serie["IBCR_cap71"], color=VERDE, linewidth=2.2, marker="o", markersize=6, label="IBCR")
for _, f in gl_serie.iterrows():
    ax.annotate(f"{f['GL_cap71']:.3f}", (f["anio"], f["GL_cap71"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=GRIS_TEXT)
ax.set_ylim(-0.1, 1.1)
ax.set_xticks(gl_serie["anio"])
ax.legend(frameon=False, fontsize=9, loc="center right")
estilo(ax, "Capítulo 71 en Colombia: Grubel-Lloyd e IBCR, 2021-2025", "GL: 0 = interindustrial puro, 1 = intraindustrial puro", "Trade Map (ITC), canasta HS2 del curso.", eje_y="índice")
guardar(fig, "g4_grubel_lloyd")
gl_serie

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 8. Crecimiento de las importaciones y participación de Colombia en cada destino

`cuota_en_socio_pct` = exportaciones de Colombia al destino / importaciones totales de HS 710391 del destino. `crec_importaciones_socio_5a_pct` es el crecimiento de esas importaciones totales.

In [ ]:
mercados = destinos_2025.nlargest(15, "valor_kusd")[["pais", "valor_kusd", "cuota_en_socio_pct", "ranking_socio", "crec_valor_5a_pct", "crec_importaciones_socio_5a_pct"]]
exportar(mercados, "g4_mercados")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
orden = mercados.dropna(subset=["cuota_en_socio_pct"]).sort_values("cuota_en_socio_pct")
axes[0].barh(orden["pais"], orden["cuota_en_socio_pct"], color=AZUL, height=0.6)
for y, (v, r) in enumerate(zip(orden["cuota_en_socio_pct"], orden["ranking_socio"])):
    axes[0].annotate(f"{v:.1f} % (n.º {r:.0f})", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(axes[0], "Cuota de Colombia en las importaciones de cada destino, 2025", "Quince mayores destinos", eje_x="%", rejilla="x")
puntos = mercados.dropna(subset=["cuota_en_socio_pct", "crec_importaciones_socio_5a_pct"])
tamanos = puntos["valor_kusd"] / puntos["valor_kusd"].max() * 900 + 40
axes[1].scatter(puntos["cuota_en_socio_pct"], puntos["crec_importaciones_socio_5a_pct"], s=tamanos, color=AZUL, alpha=0.55, edgecolor="white", linewidth=1.5)
for _, p in puntos.iterrows():
    axes[1].annotate(p["pais"], (p["cuota_en_socio_pct"], p["crec_importaciones_socio_5a_pct"]), textcoords="offset points", xytext=(6, 4), fontsize=8.5, color=GRIS_TEXT)
estilo(axes[1], "Cuota de Colombia frente al crecimiento de las importaciones del destino", "Tamaño = valor exportado por Colombia en 2025", "Trade Map (ITC), 2025.",
       eje_x="Cuota de Colombia (%)", eje_y="Crecimiento de las importaciones del destino, 5 años (%)")
plt.tight_layout()
guardar(fig, "g4_mercados")
mercados

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 9. Ficha financiera de las comercializadoras (EMIS)

Los tres reportes EMIS (`Esmeraldas de los Andes S A S.xlsx`, `Esmeraldas Mining Services S.A.S.xlsx`, `Esmeraldas Santa Rosa S.A.S.xlsx`) tienen restricción de redistribución y no están en el repositorio. En Colab, esta celda pide subirlos; si no los subes, la sección se salta y el resto del cuaderno funciona igual.

Trampas del formato EMIS: un aviso legal en la primera fila, encabezados reales en la fila 10, varias tablas apiladas en la misma hoja (`Estado de Resultados`, `Balance General`...), cifras en **miles de COP** y un 2021 en ceros que en realidad es dato ausente.

In [ ]:
def leer_emis_anual(ruta_archivo):
    """Extrae el Estado de Resultados de la hoja 'Anual' de un reporte EMIS. Devuelve formato largo en millones de COP."""
    hoja = pd.read_excel(ruta_archivo, sheet_name="Anual", header=None)
    empresa = str(hoja.iloc[2, 1]).strip()
    etiquetas = hoja.iloc[:, 1].astype(str).str.strip()
    inicio = etiquetas.index[etiquetas == "Estado de Resultados"][0]
    fin = etiquetas.index[(etiquetas == "Balance General") & (etiquetas.index > inicio)][0]
    anios = [str(c)[:4] for c in hoja.iloc[inicio, 2:7]]
    bloque = hoja.iloc[inicio + 2:fin, 1:7].copy()
    bloque.columns = ["cuenta"] + anios
    bloque = bloque.dropna(subset=["cuenta"])
    largo = bloque.melt(id_vars="cuenta", var_name="anio", value_name="miles_cop")
    largo["anio"] = largo["anio"].astype(int)
    largo["miles_cop"] = pd.to_numeric(largo["miles_cop"], errors="coerce").replace(0, np.nan)   # 0 = no reportado
    largo["millones_cop"] = largo["miles_cop"] / 1000
    largo.insert(0, "empresa", empresa)
    return largo

base_busqueda = RUTA_BASE if MODO == "local" else "/content"
archivos_emis = sorted(glob.glob(os.path.join(base_busqueda, "**", "Esmeraldas*.xlsx"), recursive=True))
if not archivos_emis and MODO == "github":
    from google.colab import files
    print("Sube los tres reportes EMIS (o cancela para saltar esta seccion).")
    files.upload()
    archivos_emis = sorted(glob.glob(os.path.join("/content", "**", "Esmeraldas*.xlsx"), recursive=True))

if archivos_emis:
    emis = pd.concat([leer_emis_anual(a) for a in archivos_emis], ignore_index=True)
    CUENTAS = ["Total Ingreso Operativo", "Ganancia operativa (EBIT)", "EBITDA", "Ganancia (Pérdida) Neta"]
    resumen_emis = emis[emis["cuenta"].isin(CUENTAS)].pivot_table(index=["empresa", "anio"], columns="cuenta", values="millones_cop")[CUENTAS]
    resumen_emis["margen_EBIT_pct"] = resumen_emis["Ganancia operativa (EBIT)"] / resumen_emis["Total Ingreso Operativo"] * 100
    exportar(resumen_emis.reset_index(), "g4_emis_resumen")
    display(resumen_emis.round(1))
else:
    emis = None
    print("No hay reportes EMIS disponibles: se salta la seccion.")

In [ ]:
if emis is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    empresas = resumen_emis.index.get_level_values("empresa").unique()
    for ax, (cuenta, titulo) in zip(axes, [("Total Ingreso Operativo", "Ingresos operativos"), ("Ganancia operativa (EBIT)", "EBIT"), ("margen_EBIT_pct", "Margen EBIT")]):
        for color, empresa in zip(CATEGORIAS, empresas):
            datos = resumen_emis.loc[empresa, cuenta].dropna()
            ax.plot(datos.index, datos.values, color=color, linewidth=2.2, marker="o", markersize=5, label=empresa)
        ax.set_xticks(range(2021, 2026))
        if cuenta != "margen_EBIT_pct":
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax.axhline(0, color=GRIS_EJE, linewidth=0.8)
        estilo(ax, titulo, "millones de COP" if cuenta != "margen_EBIT_pct" else "%", eje_y="")
    axes[0].legend(frameon=False, fontsize=8.5)
    fig.text(0.01, -0.03, "Fuente: EMIS (ISI Markets), reportes de empresa, 2026. Los ceros de 2021 se tratan como dato ausente.", fontsize=8, color=GRIS_EJE)
    plt.tight_layout()
    guardar(fig, "g4_emis")

**Análisis del equipo:** ¿Qué dicen los márgenes sobre la capacidad de las comercializadoras para financiar una estrategia de valor agregado?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 10. Tablero de decisión

La ficha cierra con un tablero que reúne todos los indicadores. La tabla siguiente los junta tal como quedaron calculados arriba; el equipo la completa con la columna de **señal** y **decisión** de su cadena interpretativa (DATO → CÁLCULO → INDICADOR → INTERPRETACIÓN → SEÑAL → DECISIÓN).

In [ ]:
ultimo = concentracion.iloc[-1]
tablero = pd.DataFrame([
    {"dimension": "Concentración", "indicador": "HHI destinos 2025", "valor": ultimo["HHI"]},
    {"dimension": "Concentración", "indicador": "Theil destinos 2025", "valor": ultimo["Theil"]},
    {"dimension": "Concentración", "indicador": "Número equivalente de destinos 2025", "valor": ultimo["numero_equivalente"]},
    {"dimension": "Participación", "indicador": f"Participación del mayor destino 2025 ({participacion.iloc[0]['pais']}) %", "valor": participacion.iloc[0]["participacion_pct"]},
    {"dimension": "Desempeño", "indicador": "CAGR exportaciones 2016-2025 %", "valor": cagr(total[2016], total[2025], 9)},
    {"dimension": "Competitividad", "indicador": "RCA Colombia HS 710391", "valor": vcr.set_index("pais").loc["Colombia", "RCA"]},
    {"dimension": "Competitividad", "indicador": "RSCA Colombia HS 710391", "valor": vcr.set_index("pais").loc["Colombia", "RSCA"]},
    {"dimension": "Competitividad", "indicador": "RCA Colombia capítulo 71, 2025", "valor": vcr_cap71.iloc[-1]["RCA_cap71"]},
    {"dimension": "Estructura", "indicador": "Grubel-Lloyd capítulo 71, 2025", "valor": gl_2025},
    {"dimension": "Mercados", "indicador": "Cuota de Colombia en su mayor destino %", "valor": mercados.iloc[0]["cuota_en_socio_pct"]},
])
tablero["senal"] = ""
tablero["decision"] = ""
exportar(tablero, "g4_tablero")
tablero

**Análisis del equipo:** Complete las columnas de señal y decisión para cada indicador.

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# Entrega

1. Revisa que todas las celdas **Análisis del equipo** tengan texto.
2. `Archivo → Descargar → Descargar .ipynb`.
3. Sube el archivo al aula virtual. Las figuras y tablas de `salidas/` pueden ir al informe escrito.

# Referencias

Balassa, B. (1965). Trade liberalisation and "revealed" comparative advantage. *The Manchester School, 33*(2), 99–123.

Yu, R., Cai, J., & Leung, P. (2009). The normalized revealed comparative advantage index. *The Annals of Regional Science, 43*(1), 267–282.

Grubel, H. G., & Lloyd, P. J. (1975). *Intra-industry trade*. Macmillan.

World Bank. (2025). *World Development Indicators*. https://data.worldbank.org

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Durán Lima, J. E. (s.f.). *Indicadores de comercio exterior y política comercial: Generalidades metodológicas e indicadores básicos*. CEPAL.

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55